# Parsing object dependency using NetworkX

In [1]:
# this magic for develop only
%load_ext autoreload
%autoreload 2

In [2]:
from ggblab import GeoGebra
from ggblab.parser import tokenize_with_commas

import networkx as nx
import polars as pl
import xml.etree.ElementTree as ET
import xmlschema

In [3]:
# initialize base class, not open GeoGebra Widget
ggb = GeoGebra()

Using local cached file: xsd/common.xsd


In [4]:
# load from .ggb (zipped, and base64 encoded)
c = ggb.construction.load('2025_13_01.ggb')

In [5]:
# parse loaded xml as python dict
o = c.ggb_schema.decode(c.geogebra_xml)

In [6]:
# check command part in archive file, and found not enough information for construction...
for _c in o["command"]:
    _ci = tuple(zip(*_c['input'].items()))[1]
    _co = tuple(zip(*_c['output'].items()))[1]
    print(f"{', '.join(_co)} = {_c['@name']}[{', '.join(_ci)}]")

poly1, f, g, h, i, E, D = Polygon[C, A, 4]
O = Midpoint[C, A]
c_1 = Circle[O, C]
P = Point[c_1]
j = Ray[C, P]
k = Ray[A, P]
l = OrthogonalLine[E, k]
m = OrthogonalLine[D, j]
n = Line[D, j]
p = Line[E, m]
q = Line[A, l]
r = Line[C, k]
poly2, s, t, a_{3}, b_1, G, H = Polygon[A, C, 4]
e = Ray[C, A]
P' = Point[c_1]
d = Ray[A, P']
B = Intersect[d, j]
f_1 = Line[C, d]
g_{4} = OrthogonalLine[C, d]
w = Vector[C, B]
c_{5} = Circle[C, 1]
I,  = Intersect[c_{5}, e]
J, K = Intersect[c_{5}, f_1]
u = Vector[C, I]
v = Vector[C, K]
h_1 = OrthogonalLine[B, e]
L = Intersect[h_1, e]
γ = Angle[A, C, B]
β = Angle[A, B, C]
α = Angle[C, A, B]
θ = Angle[Vector[C, A], Vector[A, B]]
b = Distance[C, A]
c = Distance[A, B]
u_1 = Translate[Vector[(((u * w)) / ((u * u)) * u)], C]
v_1 = Translate[Vector[(((w * v)) / ((v * v)) * v)], P']
δ = Angle[C, P, A]
ε = Angle[C, P', A]
i_1 = Segment[P, O]
t1, p_1, o, c_3 = Polygon[O, C, P]
t2, p_2, o_1, a_1 = Polygon[O, A, P]
t3, p_3, a_2, c_4 = Polygon[A, C, P]
poly3, j_1, k_1,

## with Applet

In [7]:
r = await ggb.init()

In [9]:
r = await ggb.function("setBase64", [ggb.construction.base64_buffer.decode('utf-8')])

In [10]:
# compare in archive and in applet
for _c in o["command"]:
    _ci = tuple(zip(*_c['input'].items()))[1]
    _co = tuple(zip(*_c['output'].items()))[1]
    c1 = (f"{_c['@name']}({', '.join(_ci)})"
          .replace('OrthogonalLine', 'PerpendicularLine')
          .translate(str.maketrans('[]', '()')))
    c2 = await ggb.function("getCommandString", [_c['output']['@a0']])
    if (c1 != c2):
        # print(f"{', '.join(_co)} = {c1}")
        print(f"{_c['output']['@a0']}: {c1}, {c2}")

u_1: Translate(Vector((((u * w)) / ((u * u)) * u)), C), Translate((u w) / (u u) u, C)
v_1: Translate(Vector((((w * v)) / ((v * v)) * v)), P'), Translate((w v) / (v v) v, P')
w': Translate(Vector((Γ_{uu} * u) + (Γ'_{uu} * v)), C), Translate(Γ_{uu} u + Γ'_{uu} v, C)


In [11]:
for _e in o["expression"]:
    c1 = (_e['@exp']
          .translate(str.maketrans('[]', '()')))
    c2 = await ggb.function("getCommandString", [_e['@label']])
    if (c1 != c2):
        print(_e['@label'], c1, c2)

proj_{u}w ((w * u)) / ((u * u)) (w u) / (u u)
proj_{v}w ((w * v)) / ((v * v)) (w v) / (v v)
Γ'_{uu} (-(((proj_{u}w * cos(θ)) - proj_{v}w) / (sin(θ))^(2))) -((proj_{u}w cos(θ) - proj_{v}w) / sin²(θ))
Γ_{uu} (proj_{u}w - (proj_{v}w * cos(θ))) / (sin(θ))^(2) (proj_{u}w - proj_{v}w cos(θ)) / sin²(θ)
text1 "1.  Thales's  theorem: right  triangle  inscribed  in  a  circle" None
text3 "3.  Pythagorean  theorem" None
text3_{1} "$\|a\| = \|b\| \cos(\gamma)+ \|c\| \cos(\beta)$
$\|b\| = \|c\| \cos(\alpha) + \|a\| \cos(\gamma)$
$\|c\| = \|a\| \cos(\beta) + \|b\| \cos(\alpha)$" None
text5 "5.  Projection" None
text4 "4.  Orthocenter  and  Law  of  cosines" None
text3_{2} "$\|a\|^2 = \|b\|^2 + \|c\|^2 - 2 \|b\|\|c\| \cos(\alpha)$
$\|b\|^2 = \|c\|^2 + \|a\|^2 - 2 \|c\|\|a\| \cos(\beta)$
$\|c\|^2 = \|a\|^2 + \|b\|^2 - 2 \|a\|\|b\| \cos(\gamma)$" None
text5_{1} "$\|b\| \cos(\theta) = \frac{\vec{b}\cdot\vec{a}}{\vec{a}\cdot\vec{a}}$" None
g_3 ((Area(t5) * g_2) * g_2) Area(t5) g_2 g_2
text2 "2.  Geometri

In [12]:
## not working... 
# ft = {}
# for e in o['element']:
#     _n = e['@label']
#     cmd = None
#     exp = None
#     for _c in o['command']:
#         _ci = tuple(zip(*_c['input'].items()))[1]
#         _co = tuple(zip(*_c['output'].items()))[1]
#         if _n in _co:
#             ci = _ci
#             co = _co
#             cmd = (f"{_c['@name']}({', '.join(_ci)})"
#                   .replace('OrthogonalLine', 'PerpendicularLine')
#                   .translate(str.maketrans('[]', '()')))
#             break
#     for _e in o['expression']:
#         if _n == _e['@label']:
#             exp = _e['@exp']
#     if (e['@type'] != 'text') and (cmd or exp):
#         print(_n, e['@type'], tokenize_with_commas(cmd or exp))

In [13]:
# build construction protocol
construction = {}
for o in await ggb.function("getAllObjectNames"):
    r = await ggb.function(["getObjectType", "getCommandString", "getValueString", "getCaption", "getLayer"], [o])
    construction[o] = r

In [14]:
ggb.parser.initialize_dataframe(df=pl.DataFrame(construction, strict=False))

In [15]:
ggb.parser.df

Name,Type,Command,Value,Caption,Layer
str,str,str,str,str,i64
"""C""","""point""",null,"""C = (0, 0)""",null,9
"""A""","""point""",null,"""A = (2.2, 0)""",null,9
"""poly1""","""polygon""","""Polygon(C, A, 4)""","""poly1 = 4.6""",null,3
"""f""","""segment""","""Segment(C, A, poly1)""","""f = 2.2""",null,2
"""g""","""segment""","""Segment(A, E, poly1)""","""g = 2.2""",null,2
"""E""","""point""","""Polygon(C, A, 4)""","""E = (2.1, 2.2)""",null,2
"""D""","""point""","""Polygon(C, A, 4)""","""D = (0, 2.2)""",null,2
"""h""","""segment""","""Segment(E, D, poly1)""","""h = 2.2""",null,2
"""i""","""segment""","""Segment(D, C, poly1)""","""i = 2.2""",null,2


In [16]:
ggb.parser.parse()

In [17]:
nx.write_network_text(ggb.parser.G)

╟── C
╎   ├─╼ poly1 ╾ A
╎   │   ├─╼ f ╾ C, A
╎   │   ├─╼ g ╾ A, E
╎   │   ├─╼ h ╾ E, D
╎   │   └─╼ i ╾ D, C
╎   ├─╼ E ╾ A
╎   │   ├─╼ l ╾ k
╎   │   │   └─╼ q ╾ A
╎   │   ├─╼ p ╾ m
╎   │   │   └─╼ V ╾ n
╎   │   │       ├─╼ poly7 ╾ M
╎   │   │       │   ├─╼ d_2 ╾ V, M
╎   │   │       │   ├─╼ e_2 ╾ M, W
╎   │   │       │   ├─╼ f_3 ╾ W, Z
╎   │   │       │   └─╼ g_{6} ╾ Z, V
╎   │   │       ├─╼ W ╾ M
╎   │   │       │   └─╼  ...
╎   │   │       ├─╼ Z ╾ M
╎   │   │       │   └─╼  ...
╎   │   │       └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ D ╾ A
╎   │   ├─╼ m ╾ j
╎   │   │   └─╼  ...
╎   │   ├─╼ n ╾ j
╎   │   │   └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ O ╾ A
╎   │   ├─╼ c_1 ╾ C
╎   │   │   ├─╼ P
╎   │   │   │   ├─╼ j ╾ C
╎   │   │   │   │   ├─╼ B ╾ d
╎   │   │   │   │   │   ├─╼ w ╾ C
╎   │   │   │   │   │   │   ├─╼ u_1 ╾ u, C
╎   │   │   │   │   │   │   └─╼ v_1 ╾ v, P'
╎   │   │   │   │   │   ├─╼ h_1 ╾ e
╎   │   │   │   │   │   │   └─╼ L ╾ e
╎   │   │   │   │   │   │       ├─╼ t_1 ╾ C
╎   │   │   │   │ 

In [18]:
ggb.parser.parse_subgraph()

found: 'C' => 'c_{5}'
found: 'A', 'C' => 'E'
found: 'A', 'C' => 'D'
found: 'A', 'C' => 'e'
found: 'A', 'C' => 'O'
found: 'O' => 'c_1'
found: 'c_{5}', 'e' => 'I'
found: 'c_1' => 'P''
found: 'c_1' => 'P'
found: 'P'' => 'd'
found: 'P' => 'F'
found: 'P' => 'j'
found: 'P' => 'M'
found: 'P' => 'h_3'
found: 'P' => 'k'
found: 'j' => 'm'
found: 'j' => 'n'
found: 'h_3' => 'O''
found: 'd' => 'f_1'
found: 'j', 'd' => 'B'
found: 'B' => 'h_1'
found: 'f_1' => 'K'
found: 'O'' => 'I_1'
found: 'O'' => 'd_3'
found: 'm' => 'p'
found: 'p' => 'V'
found: 'd_3' => 'A_1'
found: 'A_1' => 'k_3'
found: 'A_1' => 'l_3'
found: 'k_3' => 'r_3'
found: 'l_3' => 'B_1'
found: 'r_3' => 'C_1'


In [20]:
nx.write_network_text(ggb.parser.G2)

╟── C
╎   ├─╼ c_{5}
╎   │   └─╼ I ╾ e
╎   ├─╼ E ╾ A
╎   ├─╼ D ╾ A
╎   ├─╼ e ╾ A
╎   │   └─╼  ...
╎   └─╼ O ╾ A
╎       └─╼ c_1
╎           ├─╼ P'
╎           │   └─╼ d
╎           │       ├─╼ f_1
╎           │       │   └─╼ K
╎           │       └─╼ B ╾ j
╎           │           └─╼ h_1
╎           └─╼ P
╎               ├─╼ F
╎               ├─╼ j
╎               │   ├─╼ m
╎               │   │   └─╼ p
╎               │   │       └─╼ V
╎               │   ├─╼ n
╎               │   └─╼  ...
╎               ├─╼ M
╎               ├─╼ h_3
╎               │   └─╼ O'
╎               │       ├─╼ I_1
╎               │       └─╼ d_3
╎               │           └─╼ A_1
╎               │               ├─╼ k_3
╎               │               │   └─╼ r_3
╎               │               │       └─╼ C_1
╎               │               └─╼ l_3
╎               │                   └─╼ B_1
╎               └─╼ k
╙── A
    └─╼  ...


In [21]:
labels_map = {}
for n in ggb.parser.ft:
    if n in ggb.parser.G2:
        labels_map[n] = f"[{n}: {len(nx.descendants(ggb.parser.G, n))}]"

In [22]:
nx.set_node_attributes(ggb.parser.G, labels_map, "label")
nx.write_network_text(ggb.parser.G, with_labels="label")

╟── [C: 141]
╎   ├─╼ poly1 ╾ [A: 140]
╎   │   ├─╼ f ╾ [C: 141], [A: 140]
╎   │   ├─╼ g ╾ [A: 140], [E: 13]
╎   │   ├─╼ h ╾ [E: 13], [D: 13]
╎   │   └─╼ i ╾ [D: 13], [C: 141]
╎   ├─╼ [E: 13] ╾ [A: 140]
╎   │   ├─╼ l ╾ [k: 3]
╎   │   │   └─╼ q ╾ [A: 140]
╎   │   ├─╼ [p: 8] ╾ [m: 9]
╎   │   │   └─╼ [V: 7] ╾ [n: 8]
╎   │   │       ├─╼ poly7 ╾ [M: 9]
╎   │   │       │   ├─╼ d_2 ╾ [V: 7], [M: 9]
╎   │   │       │   ├─╼ e_2 ╾ [M: 9], W
╎   │   │       │   ├─╼ f_3 ╾ W, Z
╎   │   │       │   └─╼ g_{6} ╾ Z, [V: 7]
╎   │   │       ├─╼ W ╾ [M: 9]
╎   │   │       │   └─╼  ...
╎   │   │       ├─╼ Z ╾ [M: 9]
╎   │   │       │   └─╼  ...
╎   │   │       └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ [D: 13] ╾ [A: 140]
╎   │   ├─╼ [m: 9] ╾ [j: 39]
╎   │   │   └─╼  ...
╎   │   ├─╼ [n: 8] ╾ [j: 39]
╎   │   │   └─╼  ...
╎   │   └─╼  ...
╎   ├─╼ [O: 122] ╾ [A: 140]
╎   │   ├─╼ [c_1: 119] ╾ [C: 141]
╎   │   │   ├─╼ [P: 109]
╎   │   │   │   ├─╼ [j: 39] ╾ [C: 141]
╎   │   │   │   │   ├─╼ [B: 27] ╾ [d: 34]
╎   │   │   │   